# ARSO high-water thresholds

Parses `arso_vv.xml` (ARSO station data with the first, second and third high-water discharge thresholds) into `arso_vv.parquet`, which `performance_graphs.ipynb` uses.

In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET

def parse_arso_xml(xml_file):
    # Parse XML file
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Initialize lists to store data
    data = []
    
    # Iterate through each station
    for postaja in root.findall('postaja'):
        station_data = {
            'sifra': postaja.get('sifra'),
            'longitude': postaja.get('wgs84_dolzina'),
            'latitude': postaja.get('wgs84_sirina'),
            'kota_0': postaja.get('kota_0'),
            'reka': postaja.find('reka').text if postaja.find('reka') is not None else None,
            'merilno_mesto': postaja.find('merilno_mesto').text if postaja.find('merilno_mesto') is not None else None,
            'ime_kratko': postaja.find('ime_kratko').text if postaja.find('ime_kratko') is not None else None,
            'datum': postaja.find('datum').text if postaja.find('datum') is not None else None,
            'vodostaj': postaja.find('vodostaj').text if postaja.find('vodostaj') is not None else None,
            'pretok': postaja.find('pretok').text if postaja.find('pretok') is not None else None,
            'prvi_vv_pretok': postaja.find('prvi_vv_pretok').text if postaja.find('prvi_vv_pretok') is not None else None,
            'drugi_vv_pretok': postaja.find('drugi_vv_pretok').text if postaja.find('drugi_vv_pretok') is not None else None,
            'tretji_vv_pretok': postaja.find('tretji_vv_pretok').text if postaja.find('tretji_vv_pretok') is not None else None,
            'pretok_znacilni': postaja.find('pretok_znacilni').text if postaja.find('pretok_znacilni') is not None else None,
            'temp_vode': postaja.find('temp_vode').text if postaja.find('temp_vode') is not None else None
        }
        data.append(station_data)
    
    # Convert to DataFrame
    df = pd.DataFrame(data)
    
    # Convert numeric columns
    numeric_columns = ['longitude', 'latitude', 'kota_0', 'vodostaj', 'pretok', 
                      'prvi_vv_pretok', 'drugi_vv_pretok', 'tretji_vv_pretok', 'temp_vode']
    
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

# Read and process the XML file
xml_file = 'arso_vv.xml'  # Update with your XML file path
df = parse_arso_xml(xml_file)

# Display the first few rows of the DataFrame
print("First few rows of the processed data:")
print(df.head())

# Display basic information about the DataFrame
print("\nDataFrame Info:")
print(df.info())


In [ ]:
df[['station_id', 'vv1', 'vv2', 'vv3']] = df[['sifra', 'prvi_vv_pretok', 'drugi_vv_pretok', 'tretji_vv_pretok']]
df[['station_id', 'vv1', 'vv2', 'vv3']].to_parquet('arso_vv.parquet')

In [ ]:
import os

# Directory path
dir_path = "data/ARSO_podatki_2/20250110/Hidro/simHPS_Q_2021-23"

# List all files in the directory
file_names = os.listdir(dir_path)

# Extract the first 4 digits from each file name (if present)
station_ids = []
for fname in file_names:
    # Take first 4 characters if they are digits
    if len(fname) >= 4 and fname[:4].isdigit():
        station_ids.append(fname[:4])

from collections import defaultdict

# Group station_ids by their first digit
groups = defaultdict(list)
for id in station_ids:
    if id:  # make sure id is not empty
        groups[id[0]].append(id)

# Print each group on one line
for first_digit in sorted(groups.keys()):
    print(','.join(groups[first_digit]))
